# Forecasting monthly groundwater levels in Bangladesh
**13:30–14:45 · Hands-on session 4**

Does filling gaps improve a six-month forecast at Haluaghat? Does satellite water storage add useful information? Compare both against the same seasonal baseline.

Open this notebook in Colab and choose **Runtime → Run all**. Small exercise-specific ZIPs open automatically, including when this notebook is downloaded on its own. No Drive mount, repository clone, or earlier session is needed. Internet access is required to install packages and download the pretrained weights. The default runs three forecast windows on CPU; GPU fine-tuning is optional.

<details>
<summary>The experiment in one example</summary>

Imagine standing at well MY050 in Haluaghat at the end of December 2012. January through June 2013 are still ahead. The historical record lets us recreate that decision and later check every forecast against the readings that actually arrived.

The preprocessing session asked what a missing reading might have been. The groundwater-storage session asked how water storage varied across a map. Here the unknown is a future reading at a particular well. Keeping that target fixed lets us ask what each earlier session contributes.

</details>

In [ ]:
from time import perf_counter
setup_start = perf_counter()
model_name = "Chronos-2 Small"  # Choose "TiRex-2" and rerun all cells to switch families.
%pip -q install "darts[torch]" huggingface_hub scikit-learn
if model_name == "TiRex-2":
    %pip -q install tirex-2
install_seconds = perf_counter() - setup_start

**During setup** · The [TSFM.ai model catalog](https://tsfm.ai/) lists which models accept past-only covariates. At Haluaghat, the practical question is simple: which satellite readings would already have arrived on the day of the forecast?

## 13:30 · Meet the well
The original monthly BWDB table keeps missing readings visible. The signed levels follow the workshop convention: a more negative value means a deeper water table. We keep a regular monthly calendar even when a reading is missing.

<details>
<summary>Reading the well record and the seasonal baseline</summary>

A level of −7 m represents a deeper water table than −3 m. A downward movement on the plot therefore means the water table deepened. An empty month represents a missing measurement; replacing it with zero would create an apparent rise to ground level.

The seasonal forecast copies each month from the preceding year. For March 2013 it uses March 2012, even if the February 2013 outcome is available later in the retrospective record. This gives a simple test of how much a model gains beyond repeating the annual pattern.

</details>

In [ ]:
#@title Automatic exercise data (expand to inspect)
from functools import lru_cache
from pathlib import Path
import base64
import io
import zipfile

# Each compact ZIP stays independent. Opening one never extracts the others.
# The portable copies let a downloaded notebook run before its branch is published.
portable_zips = {
    'fine_tuning': 'UEsDBBQAAAAIAFchdV3v98o+LC0AADaEAAARAAAAZWFybGllcl93ZWxscy5jc3ZdncuuLL1ynOfnWXo3eL8MJZ+BLcMwYGuioQd+A78/VMwvgtXbgqCttf5eZBUvwcjISPY//8//+7+f//EfqafPP/9rmvnzb/+SUo7f9M//+reU5udf/2eq7fO//yOnfj6z1vP78///939POX3+y7+nuj7/+i8pj88//1ua7R95r/Qn5ed/P3/qN5fPn/EtdfbPn/Kdszz/Zv3cv7P2z/m57/hxnd/m/G3lfKx+V41/0j6fqatEGz3Prk6KOqn7dDL5o9Loo54/frqekz7GnvGLttXLjIdofOI8qv9wnG72vt1UdfP8fn5bLfH/z3I+O2c7ba7nNePHss6PaTy/zeV5+XI6OJ95mn/G6flIm4Of1HhT46ue1lc8Q3uepdP8+XF+ebA+4j0KY1fjl+V5oPgxr2i283f9o+a7mu/x1vM7GNK8aLLl+ON0ukvfHR2kzCPQ/No8keeCB2vxQ00eohHd5O9mvtdYZwzj/c/MxQu200VLK/6Zmir6KDGmc64YxTIGE13U+lTrLUbnWTaFp588TOJpSgxW+uZ5/m9ZLLl0umjftFiA8Vx98nd1aqWon6V+6ohuRjzx1qezRm48P6bvyDM6S73E8nq6zUUdap0OPrH0WPMO1lY3jOt52aeXEuMSPz2v8owf05FiwX3+rO+OwXqePQbnLPnn46nFi/RB0zm56czaXkUPr4dYWurzk2PFnB5pXTtz6+ljnaQzM9oT3g2ZnV3U5NNFik+M6Z5iss9gPG+QpvZ7fDRW1v7W+Hylo9l5ss3KXVvdFHWzSmN5l/NTqY1lGxtr9DNMebMVW4vN1mM2KjOdeozWir/KeTF0MVr5BamS2FKrMXObNa6NljoTqT25Gr9uZ1oMaQ9eNL1q9DE1XvkFqRZbbmjrsFrYSP3bC100TdcCD2PpxUYTQGm9bmYx+T0MULuM08Ngw81YCbGixjenWMwLDNk1cKoHbD2zRfP8tz7pUW0bn2KvDe2GKjR9XrczcpUX8K+nFsc4z980KB4i3vYZET9/11zr8ZfeTru0s+n75AX6PTCeBTUDP9jrB7K0pob2tabG72KUmjlQe+92/nNj/eSDTO3b+wGhxBA943C22zmSWoC4NkdnPbEVn6NP7RunamJQNMfrdxc8UDwDQWqNRVVpgiMpg1lMcAGHBMhnL6qbdTf5ivUJCDz/jeGK5y7rbI0VWzyVmNgaAPIgY4sPztR+gCQeZnJ85xeiWozr9vHA+yQd7E+fgbS5v+C8vwVISkLBLRRc2ubLG0NY9cxeYYLLZHsJcXYM1j5DdRFYw5+io17eFXdwQgA3vUDUzWUiD0M5oB599tbuCB4Mjs2RBwMDtnzY3oPXKQE/NWsO8qUePHssy+fJ5gsc7SvcaFmbWqdXbH0a1ynMxn7+/Pkvp/3yQlMfoEZOXstwACBIu25VgbdwbH5iWyd1XAZ/Om7rfvoN/CzGGkrDefWAj2dME7LjOcoGkFr5iw4kTuA23QOQ1A5rO39WA5vzKKBPbPPJVmEPsfUDofNZ+HEMrs2L9NzUbFOzHpazetuzg/vdRQ/F0QOPWPVzMCb5oMrT9OAvOpCn4cwA0kz3+QOS4pFaLIfG4PTzuOthDpxGrPUDGeeZkthMjR93is9s/rDteKG11P7QDKSsBbREeVkQndebSWMPE9nCD95kdzB2a+1544Dld64nb9JFjJ+n3gVUPqj57K6UABsTys45mk23eaKhM1GtLlp9Gjl4GWjeyg/BOyADcOZortVAmzHiv+4cA/c8iZq7lIhFEw9o6iS2ergWZ0wy5eqKE0a8SbAonRmZdi/GpMpYPRgTYyR6lAtrvIr95DLV9HTXg4b1gQNyato86JxLz/LiiCuJ2Wa998DWAd7Gqz+zd2DrGZfRCmd6jMuKY8YxwjOm6sQbNWmjDgEFvHqKoPx/O3Up5DmAoHVNCFQdhhXWeX1xpk4CtXioVEQ143xp8IIMo2p1+DQ+GzWGsWZCJo7FtuFc7sLvABcdh6+x1I2L+n2pF0qeJ5+K1opW+/LL8ko6WmMtBPd3bxd49HkOrWfmtTTOKTlFxh50YL8dUMsHzPkbt5rqX/DQtvowCg2hUEsC/Pri0LpspHhW6ET00VO5FYQM3mvcF7kIVM54HpBbFzDns0mZjclhF8utxuo5bcS89QUTSMSdLX56HlntXwTavwjUIvw9MRXjv8vv6bozA9NrdORoVUdwIJCpm3oxJVLMClRdaqcIfOn8g4UkDZQopOP+maUVzNs4IBSjtwYUKV6y8Q4DROqElVrjz8MXaB/BZipLre0LaXq86PAsZAVwsKCqiC/HUfI0MIr3ZHyq0hzQE0gKdsOCgkW2FE+bavwzOUVGRN+HEYKjM55y1qUTPgDTD3vR52KANkQRR8vCrwGWmtanKp75y49nlkSytkb2Rl9b/KYTTUHmt0CT6ORh4YWBIrzqYvlp6rSJNtsP1FTOethzX1BjiBiEtYGYQcE7C/Ic5AXuHVOc0vgwi0XNl8uYYktaF8o6RJ9dDcf1kSo5yPhSXkJQxTRzUNJnELK6MLb00JuCnTeNfBX6DsUpu1kNYAuXd6iIv+OU0Omm5oGVHuzi7viDXLHjF+pMZyUEu+gR5sZHOVzEYNuIhd7mUMvdDz55xkIQekGQoVgMV/dRJ3lsfeCaBNLC3YzC1H6CqhWjPAUGCRxP8dbtLohzHApbFdv2ZCGAP0CoSU3NG0By8DbYeJnLQcsBm/jhQdujBcRnR2dvz1m0Rm5Ed7axWnYYJbY+FOTs/gofNVTEwwZY8eU+YrSUf4hTugNnraS9tMYB6hzC0egiX3Qx40tXvXxoAeKPTmz3kXg60ZD2Kj6DUGYwQnEiO1JDS0Jmgg1NWl8EwLPwNkUMRy1fiOmAfLagJoThUO9nfJr2/BjwjSpA6YzQ9mvHjs/Jz341nlHe0Kwt1nC8TYlHSxx9+lgaKC9PKBLgtcDqhj50mu4/0g4qcOLBjJKNtSoNr3HM7yHKIX24d53QY/6yAZOo3tTV5TcVsTYlPr9Fma11r3cGc8TB58NHBcjFuNCG6AxK+PMzWhyz3V8EaqEWN3GbIdEuXnYaUsGJQMMiMZcoZKPGACfCqdVvH2Y3oRXNCKOe2UfRDZFtKdQh8HnmRUvvPG0WlyUwCKhTP8D2XurFkHQW32RwniGUcBP0icBDMXn2cdbjKM9MYJvvAZ1jUGpsR/Ux/CaVwc71auRJDHNKqcgic7E3y5c32QrOCP59sBWzlXfIboxVF/rahI4IZZHeONsUAzcNVkJ12uiUdZSXLDxbVs0bqc4L9hvtoxo46NHZlfyPUUkKbnLIO7SNI8cCXPktjFYRs19dKaHONgKDdrRv8AJVaZ5QZz/T1H62YUdQWRCwPbRVLlitLuLnc9bKkph26SGRzWXhTG8YO2srBZKNAElvquMs3Xm5ECa9xMf61r81MyT7EzI7y2D72JOSLFXUHBcADtauTm6M1gdh9rhx0hv+9sEhxdmNEnUY3iFjCHGJoRtLkWCDkcXQjRfQGjpMU3Bx9S5JPynYh3Jcz0ISJDmzNn3Yih6SWivL8KHOioOoxoA3iXGFZFR/syOSQGOAF4d3tqRXA7/ql/NJqHn7qGI7OcF72QgjB8AEhjxIptAT+jlG/BJp4RAu8moNdZPD5XkxtX9hTGtG4QRDszUSCwlZknTm9O9Om1S9SEPRPXS462yKINmvcrEsGixku2KahoScJZCpplxaZaOroyq2VcGMognsmuYhZW28oJaJZEsdjHRI2mmh58U+2JcX8N9jWbOICSWnNQs1Pb1dpJmBZd2y5YvAJ1Pxkumj8ko7apyzYGuRopityiy/wwU04UCW7AWj7puTG3iT6uPT+BxhOX0lhSH8LW1K6eTd3VwVu1wB+NDQgcZUXp28EYihxUSqKUsOfj5BWLNIl+1dhGz0cSGNv1WMuBuRCIumFXJh/TdDcqKRI8ZL2+ZF5n4zjimIh3oxkCUd1+JqCj3XYDAiL6mF5ixkoHP3fFSYtlUy52PUi/nY6MXN5rvNkA+eMUzKbz/wHL/bAVXJAtfJ4kaAtkzbg7mdLuYPL9OqmnrOpkxRVZBdHKV58QnAanGyu3z3Lx07IZb6uOp2jMUWLyBoG41QkCPDqJ5E/Wo3TiqYuck9Vquetbmnyrl/5pEAtgzGIjLaK8Ltc4wVOvkAjjMAPJNQgkmNzMasQy2DXoe56YmG8vDo2SGszgAwZSuVY5rbgMOGnBLXlkwOiGxD77Wy38QQthatKmObAUlOl/ltgmP3FdLAYJQjiTANE9nnEkcLtG++qlNgz3imaDnzMCI5//BNeEXqWj+NGBhp6wzaZLCLs9+NsWvDY+eQcWNuyKa4ZOUS/Dn2ZJOUUSQqil0mhfL+y2ZVntlUNxfEpCdxxBPVEXjO/MmxWRLCCzvmrGJ2TibggR86PCXXMX80cYGsTRpasxl0WdEFgNF5wqIjz9n3Jp2nZeBRSaOkub8QNrWVkiCqykJBbAr1SKLiSSHpQZGIxozPXfGzHmlnb8uLYUX68BLDIcbZpHUOGheAejKvoZAfeMF4M7HnLPLZW4LH1ty/wpVIMbRatoeeCdxZYY7r2t1ROuy96wk2bqr+wPLpZb0oNogtCS3jBRqKFVDPthfxmEnd7GDxTTx8OpHePmq8iBiRl4Y0Ngjh+kbcl7ViYJoJTFwy0zSS2wkeEnmJo6rf5quQhVldz8hz6Hbz3X26UPBaRNtS7DsnuJyRZ9MQ0LbbgaErTuosXrcnC39HaHdJq1Z26vTJHsfNpSyBXBfxUXUgZXw+SwhlfJzVngM8fDCigsyBMaxEww3Xw+R020WfuQ8+/srotHxlqiDrwlFpBltqViNDrvOpsiuHEkyXAPV5H/56BLRAk8LO5dCfEB2yEIkvNobPP9xeTQd68xZU69fBtJZkI0VGaMWx/QKPhpr/BHRrrbBgK7Hbg8xq9NIo9MH5+8BZFPfA2AcXTyKkdor+iBs8bH2dUGr6Qs+m69IsSkDE2Fmn4dlk4WkEcCs4fLxQdW5+ycPgxm/0J/GAlEnG5CNIkFAlKY/zIMfw1KuNaMan2700CTGfd1SIWtRucI/AtyrlQhOo5BS5hxFNbgFKbFvypCtWwSD1vCHc9INCXEgiHhREimj3sDySulo1kqT3gG4RFjxbhxMXsB4Wqzspt9jl1T4nkUwhQtXK2wlg328kp+zr0agld26J63SwFICn+GNOGjyJih62oqUUb59kphAOCH33CzCLLFSWYYE9kZu45bpJvqwk1kGipaRyZjlMyZjm8Don6n23fhneRH0knUYA8kAWHBlr3tiLiBKuEl7I+RwGzUAcxAcqvAU3wJhOJ0jWIsGU7HgSN1XiokrEn4oRT8whMfc+9JWjniAjGNBDTY7jiDPzWHgjdcDoYYvZyu6iGia5EasavNn+aTSWeD9SCDVFnMA2Dy0DZRuG9v+QF4jcSDqrT+3f1FuV/7J0GWTC/uPPQ1C1RadS+mY3yzLQMMp4PF6NvIpzD524gR5EvAcTJ8dqjmPCVD6gtB1VCSVJiApJrdpqrxuAzIuiGiPfkmsO1pmTQ8Ait0HIC03ppqEXsedgJK/8V2qa78elzcQ2/VxeE/OeNgrf9in7NLN/zNmdF40nPHZG+Ccbc5DjDLtCd850Tc5MrC01JPCdXqA5avbW+zVllk+oI74A2ohdZSUJigTzh8H8Siw6VTrbfqcXZrS7N/FKOT06PNmSE8nDh60TEw8nfv2bdmx2zE5v6q2sIDoDhS6XERt0ZrnFOMUWjx05loHvpeAhO0SLBVA9NDeN34aWazxqn/pHepAeKSv3TZJlaDa6XIXa78tPPbzPs6Bg41+Z+uyriW6IRHZqc0gDLZIk0004uO3ruS6xp7Vl8tWaWpCLB02awrUmy8leJIPFz4ewSc1e7yJZQGvqWYuipZ9/k4So1/KDqFXsI8hERDjLV/WjC00q5QXEuTRXCbS064KZRzsCprmhQV5yr50a036thfbKCrKDbLRjQWxxvZnQKhHIllSQsIhuu6chHrxCEtLVAFL8HENbI2leqgSqbWRQwqrcBy0G0LQKdqyCC36KNQza21CKwZrV6ZW2vdWntR9f9CbPbbtx1q7uCJyKOfvf29laan9FJ6VnipKxXb3omTcurDhNE5C+yI8qz1KaJ3/b0wNSdIep9KJm3/iGEcQ5pZCjmKBPn0UKcKq1GU4sqD/PHaxAjTfRgRzC4eca3Kbchkkx35bWE+nY+cXkuJOtgGqtG3VEjxasOZa103jXYVmElhuXo8lpx1pwNFePwLiQYw/fglYBy+3w/6ydVZvbda1JAnY1Z4pj9lTbbykG3n9lfbC2Ecofh1JseZaHcKHgy5BPblbZavn7sdT8hQcW2XCFRjPfLpFe/yQlUGPP6giVgM5BBSRojqFzO19oKIREKQpVotE6jo6yJcEs2kSf2czezC/WzHBW7Pwj/3LGWrcsGj+v4dZjSMC6pOAoWdFW6KDQcjHjVUP+Y/V5c2A5AupyrfT5zGmzKa4ZPpw4U/FOmSZgd0W/gskuBJ5gAQ6SAncNt/TZSqqpoPFjaPtEMz9+5UqQsUNobjfwqdIt081VZ29qO5iyPMR2UXh0WNblTUp1nZ3Z+s2SlIy+d132hJDuBrESYV72G+2OMW8Xphhh8h2EMMpOkXuaCDwFIwNru3wLfmbIpSpUyDlUt+yYxRUL5JmGYxHciQvmG9tEo7TpAcll3sRalU2ko7YqGNvlxZTLk5PrxUZXHHP13R6RTqymJnpQEh31IQdH9ba6PQyjVgQePfSaSbr/ifBgy6vepJxirQi2ZsGYExnmildu26Mch+L8ISdO1g0jopwoRdUyXW7d+mVs7DopziqvO7FLqzzPmL8Jj5mYTGpBwrEk6yNdbvEO/yuQYNUCnM8MNW59xBZVc2WmRszmmuzyugclHIi9stEMJeZ3P/lP7MKQDJeZob4VRRhdEZiqsUqhC0u3ffi0iOaV+El35ZuXdHRGg43slNcBgNcDIpZIbIMJeHYybMcD8+NelpQgPgC2D5YC+1Sn6tQQVssp+L2QuhOVn/vHtkzRybzpri2xlYcbXXjozdTN3Ck2kLIgNbIIFOsLNRMF9oQ1LfjLwD1ynaw5v9t0YF8Z2tVVTloSg2Wr6Qsx5P8CWsPCzebsS4FMNCAwDpjBUTCDe/65Htcum2ioCrtehrLjgOjEGfurs92GliRPbfGiHNr2g9QrUk/3JOj8rC+6nKJfy62NiofJ405bWrJc7zBsLIwc/gumrCKP0pYad26oKR7s0m1ktNzsflx/FwN4gysl9f435eQMWfcFLn9R5nlc00hnpXLmKFBkTxV7aLfmNzYs4H+UQzX9V1lWJRDIaPWHHssVImEl+HyvH+KJNj+kdtTU/mvTlxA6swoRjTOndEvPV8TiXVak9T6U7th3/t5Uj1Bky5fQdNq6eNXDmx3UqumklNBKxRUXbvqqrTi9lJzIRBpDMDHJugJgjUgF298xEMWTraEGjR6xxLoPa0l0KDmzvUgP9eRBKQONcYstfd3HTRpnJ/tP+VDMTcM1ewO44+Q6ytYmYYz9TccaXtjI1e72qh8IeFtw1zT+9jW6/Ea26eTVVPxG18UmVsLIthcyhEYpzphuGUzah4sUev47aoIvX/2wXqNT1whMd3NTzhPuuKTKxOe3srN7Yw3TctneGRHmZBUx1Sr8iDlS84aODWso89KFBfFmaxf7qFglrUfLvSrkixEqg3NTLdsRg4OPNFX7qu51AHzKniuTm6ZYTVbkeStFYzV5ptQ+cmqsFKHGHq9zUOw630GZccwcHZ3jRaIUcvZ4x9t4gRqrWwcoPkxNgIqVQtE/LigKNrFDV0KiVYtZC4+vHgwjkxg4AetQ4nHBxZUx9Tr2T3zo+iemhL1Zu579zdvEVGbJIpFZO6ktoF/FRPOEJqxN4HWS/ZhEfT9OYyKiARgX31VQtapIP7kQMjOh4S1KONiapUVKCzPMonsfvXQk6cxYTgJNPF5CfQgQS4oi3S7+0kQy9PdSy37sxqNCcznGisLtOO6pdkBuDs2gh0zwh5Lx8cUPAhHAR/o8oJq/fGTEVhwCmHCpbBbvsj5TpWjQb9MZGoHCuD7agndma7sWv4fFkygQyOkWg1ZncmQPqXAYkc/GfQndwx9HcVJYay9iU7liU1fAzVQNZhaxlVU7O/ZfpBjCuaNdKVya6QfvpiOhNt+l3CLQVXddbzYA0SjhHsD7icfy6SqSjANQCkcRaSXoDFGO7vKgQIX6lPCc7/4GRgzaCjgTA1vXrjNNbohBx3UXWlKb19WmBCsFqHXeSbJNb6I8dU7aIvtpEaJC+xHc2ZXFAY7cR4WbArLrbcIHtvtN93SpY++1EZjpK5pxl5GEYgz54mKszD+6uZXCDS/nS2/M1o1AttYPTkjtY2nTnZx4ldFkld8i0RxYTvsvOjWYCC/pDFcj3DoKrvLNE56dYWUUM05E0SlHNhyraIR+yI4OEt97QOMqixR/JKLKWCNN2mTF10j9Bn+Ysnb/0WaGlDvwmdVSLwLGn4rR14CyrIzZIgz6LeLL1DEcgnB6GS8zytIqETd24NliDyCYTFVsifSz00A3ef1RLMEuPDn7uomHoo5DLaxEdsAkQPMN3VgDGCum2BH37HSHs76WhB+TOjJ6bWIrZ7CDiJfYrUtw1DFym+r3W7O/MJ+sv6CrK/au6sdMaa4mqoSq0DCwzImYC8IQURSwhMtMKKyI/FGFzWCuONYjdWG4woCzQMcI6xbggp41mUtM5l0VnAXO02X5U8CjYO4kwdSH8aqqFoUKyxlNL5puKkhX/nIxKUVqVHNlDHG2zUL57cLCTmaEVpC/cepwSIJQuyUnGBt06RYQ3dwiXb2DBGr2zUc7tSyZKnwy8XeaOgKYJCSaSjD0W8Lxl+Saqp/7eu8o7W+4JtOKp9+o3mULV4eRPFkrllljKgkk3ahqQ6g0FGlCpfiCCuxsLmd3ElTxnm8SQmyxuvXjDs7mBT8GzSqkUUWG/VycliX/YJHtf7ZRS2dWH7dWq0kx35LwYkEoTkT1LVhhGR7ZrXc3GfyT8RycLFpA0I8reKgmx6mjLUderthYFP+a02i9T/1cnB66idLfVTlEBOdPunqHPVjAgy+X83GjOLm017HUFhfBpNWdnXhPJHVwHTF7/YRsRc6RigKLeY2E5lhYzmLZdHYAOs8gKs04Rgfa2o9BGJBb2gGiGznKdlbEUGBauR9oWhQdzVSAyjZvcv3iONjzpq+X8k2WjrZ4F10otCgakX6vB9lCInEzeINSBGWqC4dvm6iqUe4+sRy2wNK+kGjZeJslNdEAxgrVSRG96mYSgxfFwXu+lKmKz04jl9lYJxrtTgBjBOGIgM27bl+GZh3jjNtad8Ac1ZkZUIuxFThGgcx0Flt15oUYq4KdCqF1ZGMgF3zldlfYBShijCXbjmII8vPU9oBOiFEL6brJRaeTXlbaDIJFifm+tuDXHG+PAKtJJ4IThSmaV9xmVHXlvsImsB6SU5tH7F7Qg74qJ54Pr/5aPqLuptwayXCUhh4hm3q9lenr3v2mLu4dXPTRfLZEOzhcFgvnJvuKdI1ml0nFCZSdBiM/t17+VAhnKJfolaxJjFwWstqqpPrYQkDZJWZ1WW7JNRBqr5c90cjW6mw6DSZ5s/lVfaUti8qJqdBb9W9bJTyyzQtip3syfaq+iYBqIu7/OIbARYoqXtZZd0NKFJfHwY0Dxsxp0L+6sM9Gwl3ybRjV8abyYIooXf48fgJw6tk4e36VrOU+LDSRE1DxWoPwLYT7dX20LpMRmd6DDvL41RxYBEkBxnohq+32E2ztiOPbAt7xzUAshmNImDNHDDR5c0wWRq3iwVgvYi2lwMCR7kLyJamL+W0qaSX9MrF4xDiDIgY0lVTZ+LR+ixhirgc3DUmuncAuJqTM/h1N1uH2ardz/QLJIFs8trq4UNVEa2aF5CjYs4ari4YUMwlspn7rQhxfSxgbcest3rpSMQLdwyX8Ti8BwElL+KPcgMbGjupsu6uMUSr6Hh6xV6zaiGKDqKh5A8bT7/cgxzYUVogPE/46RU5llJo1ROEkPx4XOI0lKme1cLeRsNTyl+6jSEl103ppR8X7xSlsuvOrVHJBj13KZC3Him92RPpfdeDvmy6XGjZCVSHUxs0ow2qSuIK+tcyVdBCzjeR2S1L6JHyYRTS/QZVk1BTR6S6iwLPGal1yNiffvhXLugvJMh7cqZpPkGyodQtSmCaiWIHHikslKDbfLIihrN6+pItNikjk557b+KSow+/R74DBCKuq0iiY6DFcmw2pe7P6QNyJV+OSM8QEpI2h4rOiDm7hexBcrgnMkglNGKcNM3uynpqi3KVcYFVGITVdQjn9/MamJhRqvmxGw1DqqxIWZHmOP8JRTiYtrb2UgVTbN4uGBXPi9UpCU1wYVLZ2qdJ96yZO3UJFhpobdvf+KV8wg0VHJuFXmxEDGLCdrbtgV14acnURStLsm0ojLalw00ZMdu86UrprZJM4iRrEYZOGfkx63IsvTWnfxcU92qsD/kbNlOheIZhWAfm51fKjtl6/sAhH190UXTdHZL1m9raUSJvxoLdbO5HdQhr/KCmlH9qDVZbtjWoePL1zS9Sp/o/HYpR0m8xo1yxD3FDUqrGE8R9KTuh2FiTVpXDeN4hxlQUxCEWcjmjivx3OTNvmN1MSynrRMOt6p2U03R5aUIpVMK6Re2AdrSHSqvm/Be0kD09XpU8mRb0A2i7oqAoNaW7KflOVOiS9oeZvheYslPRSowLopRzcTP4YOKzuDasM9SZFd6QqHvxOpMuecvAWXfSnmyFh1Ou9flJnjg0Eq72B+NCtzrlrRZc7NNOOxMXRj95AdZZuYONIRxXf15/fbIU4eG7XDW0ut3mvxWWj3JDXV3pkOJ+zfuka79aVgJzJpuVt+XvwbCr+pyzdtZa+Can/3bZr3Y0jWzn4VWnc14L1e88NbXO/BloueSsuwU64Dn27jSzyclFtjcUNoZLsMBLkxPftx7+YRrBOlupm/5SuMSwfEKHxe9Wpcz9iSK04BGTHTKszagAgsQFdN8leihKt5xdGVooqvskVLdwoEzSKSz6VOotXJBOmEG4RvJGAfHirmjWOTLneti5c0FNSpxvllI0orH4lBWRSiUNej9j5zA0LJL88pNyr5JWTGkGQCH6WNGu2n29tb75aYUYDRhOBSXUPxpItotOo1Ii7vzq7fSO0TxyHlXvjzk1tzdnEqdFcaKWlqXHTj+WxIE/FTOXAgX3rBJrksyXd3FfiZFaar2g9IYaaH5b75WpO64dWED+ua+VuonyJReQkiTJh7We96FBWJ1PLpv5IZbOSvYuxR7dFHok8a/U1UpM7kZIIuy6Sy1stLy93UTpJxVuWgV1Mn4TUEiV11ZX8zKjkPn8+ph/qYltZEQkeuvNMmgcq4xLHzL4mU2xyIwnZ9+erKsuFMU/Ey1DmzWNe11XssKncnUKYDNfKPyYdVMdUfFeiWr6W5Pw33ixIKb906GaJnqktukpyquRJR3kyzzpBhHp5a7kH5cEEWkuBBJayJNMOV+bpXlnYckLgaQ0itSEZ5Ye6KGAxZI03GJm+DF4rMvn6BbxLtumKazQx3DrVw5vy0hZzdkuQkcSu12XZS35OqHq3PD2wtMkai5qmPm6ufmvjQvMD8Mzml+5/GlwFVzjVHW8VeYdkvVbhIWtanViwwVtSIs2BkTNnFV4nxy4Vt5cUo07eQ4Ju1XdkhEintv3dFIMrXVQEV5UD2KTkzf/YIh0s2c4pdEovAI39cbivDm5Bt66xSE4rgO1Jefpxv27EOOy9jhmzqrBulNvyreLW93+4bDJmqhUdLWypWJ+aW8wFsn8tNfYyGXEJSBXJfYkW954n6JnyfjGp2BF3UXjiR3zNP4V8J45sjGwhc3LlbuKuQSohSS4V++QrHt2WafOVhF3RxyCY6/xmB9OruziEUPXS2C/GE1oJGX8uVZ7+1hbdYeuUAeMz3kfUFfm+YLGpSR9HNiHHC4ZD4iy3GBCFmToa8F1lCqLk8m+6A76mmKn6AkdVRXWMPFdXpABZ+HBSdkFMxRIjIb81DW3nuNyP1p2WwsPtO/Qb5iaBLM78P/Lo+ktPulC2SSgboqcqu4CIqpcLGxZRJmCpHxf+Efa58vq133iua11Kr9xJikpvav4CBhcb+vKoTQSE+ErOCMorSJI/aejYv9FdjNWcHqK/K6ayy4D0LSRbUhPi+9bRoj+AdYEtzl6UJeAenuGr7J6Z3NQCx01Dg/NnRZqwfnXvKkBX8T/jd6Rk43n6oSadA68V/fndMi+E1cvC+Uf5AhGCpq8D0ZQrpJhq//oIUaaGSrSzw1Ovy/fKFltmpPn5Inm+uiXjRT7lYOrgrW+IXaEkHOwHIFaQovKvwMKRhckxuuTmVKjetFZ+7i6VjAZD0df5XEs/kVVxQRCqgifXFrSuD4sFNS+Z9+osyZ7iKkMZcNEPbmq3K9pVVQRKfFKmPt/rUIbH570PYgbwd3gK2i+rMxHrVbIhBX6Yqc1cUNWlNGOW6gNb/7k2Gav2VoqkSOZf83chZbHsihM6sh78rDv7SHCObs+8+jDyDLuKSavpYOarqbY0YJdrqVSy684GSkW295Of3mgTp0+zHhs26s53aS0V2Si+axxXHSMlwcy5DfAD0XK7hhkYaB53pbd7vQO3sN87JYh2pQFW7Ttr4XzZy6Y0FzD4uTY5LiXMIzZIJwLdsQ9iIFj2EyjIEYOQXO8O/QYB5Cpq973CL3a9q1PbV4RJd0VDQKtSxy1a7pSPF50wNemcbIpOf25M9kDI3OJbe5oqu7j1VzdSksTPU7xovjvwo1bfvHaJVtP12NYoQuFkD8k2XUNTxRIsi33FKRybx0P85rH7xyYtNyszl+hEr68Q5Vuq670x1Ldwu+AP32/08VZxV/rw9fPa3l03DW2sxInKFn9Zhu3GqPB8smyhzJru4xoCB8lSrqZJKqNMhCyK3F5hu+pa9lDsGgnXGgUkjIkWzFsKIW+s6mHZ4knw3UURfdvVhjS1/HNQFfr0fe0M0fUvd26QyeeDiLZcXx6X0rCF7nezVOUy2xup5CmPCsG623YsNNioG63zLCEyshiVdY2lhJT6e6HokKGpcUzG5+6TW4uRCjh02W+XpaUJ09b9monGiRpwR8JIWJBFMnfsvlrdw3XaVPdgDSdp0TZBczImOo8lchbTtnxLhUsqMzdyvq9yyU3EWqR8e2SfFinWrZu7K4arzrkyfFvqJBG+h+5HZZP1t7BK5/64dhTnreI/UOKg2nMQbME4+gTAxn3Ue8VeZT6S7thvvrAqb2ccuqTtpEO16lIShWm/1xtz9JUfwXrJUVtErHXh9GI1axA4afXVLkmNviVUwtjkb+Eyi2JZNKy3y+X0AkTiTLN1E4IyDC569h8rXwnSybdz7OLSIX25Iep6pa662EZemyQbNXZlFZUx+Svf5vsFcSKuvtFKZhmBYUAlJ3tVk+YmU7thKtGDZri0cKvTQPudoytE+EskTos/duHwr521pFQOp9NwlubEvmQo3i/B7FwTPQi3aswjsslQ4zf9w7G+RDdanFBZItL726V01qD0u17Rc9wbdjMbBnlv/OSQz9Nv4deAnQxJP0VfiJBSv7lx9F6NMxCRlDIYr14ri+K+l2C8GIGpEzW7ulh9yG8yb+Ha8xu1aZk2YqeFE7ZhztkRtQXK7Y+BfXGL+IzSOzBMLdnoq7pfn6L1C25Ugu3m71HiGsOlYZiXQw7LsOu+t2ObyEDn6a9k0DXEOGEmacqJ8iAOXLHSEjK5lPIuA5drKxU/dS4UCwFb4plNG9fx6Z+VPtu28fiy89vDlWAXHeQfN1mTgYe2rvXE3z8bY+Pr+6YQ0iWixsifm4EHRfDEBZgPpjwwVFdxZU6jjKuiPoRLS14dtXcTw8uU9/drAltsVgeP2r7FEkuTHjJ+uMSMFx9AiG6BbO3K7hntjZZ2zJMcL2tFW/PFAr7Pa/GFEkR9HZ19feWzT5F33HgdhhPAsVTPvE61+EYisY3sWmd/RIYTsrNEgNf1xRdYABsUCrI1GfOnNOA8jWw9lXCRijNCrHkvzoiV7xRXtPdAnhozIwhNrWS+HK5/9fVLSQkUpMGJCL2qKvJLWEQmdyUfSeGjRn36ewtO3qsqEV4ppmx6QAcuGxVNbfxVCRnrkss8VcJJ0SAheVH7Zj/i/EMi6jty3uLxbUXPHOsyBCXeUsgkcb8Q07CbTYCSDpvauQFEEQiVWxbE8g0Im/bEesPr1E6z8j+x+fbvtq+4ej+YKOhs32NZZXUEpsnVtYK8aY+vNIemLZT08u93VMa9os9sqvBrKD6KkATtgLSEtcYF5e9JqdqktfMqnJxwuk6/3ETIVNbM5cC6cosrpoqyZM26kEhWymr9dZHh0m98mV+G6TZuRwWksbhuvl6oI9vUMNHEHxJpr989zpZ05NVYQUubpEjxWY11lUUt6y2StRDJ4P5cdKtvLtzoUfbiZC5AfFMAbbLEugoFl0I/wL5GM2r5apxx4UCWOCtZNTyOOySfAbGsmIZ+Sm433HxWNejzHWGuaO06uDgZX77auGm3Ci911UcRrkp+UeNr3+f1Ud9pPdmN+vcXou8YkCmzDu04U3KzvHW50XvqRyyn+yO4iXfFV5AFf+BwrjGyuKvORTaQ+qGW7PfCwpTu3bzN3+jX4umKKZdktDq0Qra/gWv5PDqrX41fXGBTp3vnVCKdQFWpbxUsrgv3gTCroC5f7W+r5XvGd6kOHNVDxcz326A/+gLLBY6un20PSZQ19fIg1WRNez2EndVnkUOd5HHa0gw9N6+JtIkv6msP12tnX7qRS5f3BcvVOy/5uFWlFlimhu/pXqLZyq2VbfMd4iqBJA9ybiz8YMUjbs++Cyuw6scxqos40ncoHLjJqqXvuElXc9ApcCt5f3b+kKS8791vnJ0xzdMXtBV0uzWRNbViMtXgw9dJLWj9TG7RvD5McUUX8Mk/mZGoNwaKcyXrh23ZZdgqt8lzD1VTk970qjeSWNxtbFXx+TGtcCUbBT1Vd6rAsOYVjfQFOfpCgyqB+rpBfYfNQVgJGGBLlvGUg3X6El9rKK7d85dquvZiiCa5F7syKveIT0VlJX6K7yZStqN/XTvDGK2bZq98PHGr3Xm/pcanoKb9RJVWrKeOtq1qVKrnaZqMVrv3M/Ps+AadwtuvJaOQYFGKkrINxNg5RJ0LZ4QUqzz5+tbI8p2Uqxo0MhRfNBxY3t7vFqLK9uPUSVPGv/pKGN3wvicNvoDhK3NskaZ22/BN8meJlnc8h/ULGWzrzUHEV4Gr7Xu12rqRVv1StBKX0m7Kf/gSZTij1Ht9kzlkvunlL0RI3NGpGrKw1pQWoL9+V3dm+MLQ9Y//BFBLAQIUAxQAAAAIAFchdV3v98o+LC0AADaEAAARAAAAAAAAAAAAAACAAQAAAABlYXJsaWVyX3dlbGxzLmNzdlBLBQYAAAAAAQABAD8AAABbLQAAAAA=',
    'gap_filling': 'UEsDBBQAAAAIAFchdV3JK8pxfQ4AAGcpAAAPAAAAb3RoZXJfd2VsbHMuY3N2VVrNsnOpDdzPs3hOIQEClpnMIj+LVCWzyTKLvEHevwLdLY5vKvV5fG0JEN1SS8e//+d///38/pcy7PPPv5UyPr/9o9T2+de/rfTz9zn3389///2PYuXz5z9KnZ/f/lQsPr//tYz2i5dSfi22///5tT9m7bz0+vnVnhX7jT+rf35tz/D5OR9F23+sj+9Fzoe9ulw4XMTjPrbBfEr3fr457bzfbuKYl+if862Kz/rs8LKad7mpcDOfef7phrUXlix+HO49tkEfsw94ieHnD3upuG4a3Fh5DFbbn8F4VdqU88ZpOB4v3GvYed+20UhHHY7W00dgR9iQlzjb9nOm9lSGZJ1tbl9zRwa7KSEfoTNZYCuG1fap7bzaUybfT4S74108bRwn1s+ODIeXt6FAz8og4Kqmn5c+eG/nX29nb205jlt8uynPGrmnKS+I5v7iXMcogluqnTsb58Py9MI7nPinI/7b2d3SEnxiMrhBs9norf/0ZkDI+XPGgf5mpTsrctc2ek8M/Hw6zlctEPjz8f5+R8C8YlurTkSqLh3RCOqxkYgT9qKzBVYdCIxrL3URF7qKfZUfeXFdXkVoNhgHLebSN/FXm3SutwvO7BzsuLFk2HoGbq0ukQfhag6G8Wwz5jn7vl1QI9qUB09Ir0WGVVxwY8gHuTGxfAQgFGAKQ8i7siSY2TMm3YwGuAHD4fTTgq8xAOnkmWhW05dY5tt6nsOVGfh+hTM7x9vJZDYC7Xx0jn9yAmjmTX56Hq0eL9uODAihbulw5L/jdRzEivUj95NEKxMn41UdHJHeAgJoWDdtB/y0r8s6l1JuqC7THHGaiHfHZZVDsu1xzXPr47yLfZGTZAt4siU/P7m2PWW66SKC4Zzt/LchpSQKnexv7e7pB9Umrm2vZIlJobCCMc74dS1X/BI3I/ZS7Zxu4IxlX3Rw/yCWDaQAIJaELD6VlOTlB9O2F8MuTOsH4+24hb2twhyq+lCYzjcF692Vy98K4GkWZLeBJNaApJ3fggVl4iJaax+eHHvyl3Dg6fbROhMIF10k88CpShBwpqrEsPtTh3y91BusJtWULjqPBnhWgO1Qe1OPHwzmyp1tmlxd+qHATefKBV4a0aq/eSE+T/XGUqpyBzVy9vIvQD8AYwHNDXtqHUGa3oj0IPscjnxlrF7+FQLUWRQAMkeZPkktmNpIP93qPi6wW+6mLgkH0R4uIVAzfwx6X4Bcq8hRhtwjFy/xXImOaAQKFPB9XRuNxHNMGV6mEYisMEUrbqIFXkbFsRD0gPPlBHTPHVyaFTDbQahGzVNRj+ZUoUCmC/LUaH5p1e2FekGQQAhjRCgENt/Wm5Dgp4vvLmrBCY9A5cFlK4/jk/Fsss6E6CISaGLwq6w1nDULQmH0xpTPglMLglm/GFSBoUaqjET8qVjMUx0xr5Wnaon5qDxFvQSyI0R4M8oJehmSdsKdDVKAAmLTKORymRzWi3zqsgacR0d+qv2q2P1vk8KDRivzQ142OboU6mT2yXM0uoREej6UJoeuqMVhx8gzXhItFdZeX77sy0btaqyqZHZIwCIzImIZ+lDolyFNOEA/JoBAStcK9nTUU1u8eAvZjxQtWKdekcMXorANro76t5WxrkJibMhTMioYi4OC/WFlnZlDryxgPGpDxajnX3x3XBQkqVqBNl1ScUHYIceHKakgGs5URnXYdGUvuchAqUoXuRp7l4y6alDr3GFWwJbX9rN2ncoARpoqsKSPUTMa0YhtNVNx9sgDXq3YA6nydF8obMHIhrCNSko5MZucsXxNSqL2RT5RggeyUwuZLApjHXrf+5cAj048lSVvX9oRdbUBvHWksBpAONPlznZg0UpQ5pbEuSOBgcogVShiGwvOweOREZB8cwQlWm/pQ3QL9GrQaFqT/PTGRFBVZBj/jWDuyFgAp5z1dAao7XMQSoMMMShtVv+e8jYoh6fLBYlm+9tjsPcrFAq1K6nCSyeRXNmvFmmWJl3dknJWoTSQV1EmKIXWYhJmnIuY56cyfmQ/076QN4zuKMjRzDqD2tMSTy78ZTxW5rITrlBoC+4ku86T7qW7JGFKFqMpPFvPM4lp9qZ93EfZwEb6aiSJq5811IKgO19i2hzpzdLbGNQDFKm4rLFww53VvZAphX8zFv7uujTRzE7FREJUda9Tm+TEotlXtnR6J6LPUY+rnjSz842WMNzxi2AacnL01XU2Jlsh9grpxhOJKRAbbahNJ2lWBXl0J0FteMYHnrXk7qpellSWkimGNiZsUB3SaIooxjP7AoZrTU9ffDseFpuAg2j1RKHS65xNSHP2qkLc4O/d2aXcmRdsb4OZxDk3QsIj708W6Ky5wZorldYv6Q5CoXgkXs66EuusRX2QrXGXB8nYBCxdZu8a4hTl6spiWKVIu6iVra5jQnOSY0d2npMI0Qmc3BrUg3KwEiWl8D5a9HfA4VQIZwrDC5UQyuLhkcTShsgr5rfNJl/YD8SuowE8NqoSFHpNfZvsLe27s/g1KQlxubEed1EdiYAFocqDZy4nCpfqiTGuiuLtaiq0HFNVXNLs+zswn2DyEejYeXdqQyodFoLCknlCzb4q5EnbmCpF+bo5VzRzKKTOHR5s3qwvbFZ12XFJs8DXIVW81y4dB5mmWd9SUEiVoilWXK4sRjXHNJUNa0fDg10URhNjFpn2PMeMwGp9fTLvnBoXrLpBuyqrSCvkZjZcqkIdgeUZxi3ykTOKeTetsjNwCadzaxTM0EZSHpiMhbRip9IY9w5memjSrRSGW/xIUY18r+noUopHowpf6Wqlq6MAp4ZrLjnvZ0pJXOJwVfQYInvN+hN5uJcnA6LO2Kst6uDJpD6o1UMUIUfiVpvg+Ch1r0Dt2Wk19UpxKTEQ5ZwLGXthmi0EcEmqTRiOS4d9+atx8s2OGHPgzaxoFNCcEjdywIesPVHXlOu7VPRUzZ0Umuski5PpOfBbrEBLwRqJf99ZqxK9nBssaZWdZyhimuIOD7tkyLyleTtkXfvSsO3B+coJNho4h4AEvirmMx/Zd95VZpLTD+J6mlZPKbfY5coq0mq3MAdaEJyk37rFTXIpNGp4jzwy8uNk712OLD4MOobxcYije7Ymm5k23jWNC8oHks3VSJxxdeX/ANb1zYkjnYVWZtQh7AODjeOFSkU5wJvznheHeQdmDTRMkbVD3SCrjR3I5EMPY+KeBW970ZC76NoS5juvdSZQ9QlulB6Ihmvw2bLjU+0sJi+eFbIZ2rDJu4PC3ObUaJUafRgR3Cjq5yXAkracgpirm6MgXpTYBxjlx0DoBLXL0yUDOsp3ENUGr7WbBgMSr0xJFVJULl4aDEBisaMh+QwTo3GGHJfKu1ZV2YoDLmxsTgrLYj6zM+bdSMvstuRlrruHnn4orhd07aauqvRqHwGD/GZRmGkdl4lVxYRJvSIEHInG0FMmPmSayQXfinpxTc86qV62coiyy1Gh5GOxQY6zu/WZbtwYAlMV4AC4sVJNjgBYf7OnXnKRTNgwNZahRVoSj/xWwn5h4nCv2qTm92kz7aWuKyrcbanvzoDZxV/T+FJjiZUaNzc92cDr3lDncjsXebXfab9zfnoeMTRWAOcAPNCSnPoKGq6kwA4bZbSeg92GRDkJWTl+gj80H1mJfqZguhls/XVPtmUY4JiVgQpolHRQE3fRkMO9omRU6A7joO70Cn6tdy0sTdZC/+lLcfPDJMYBT068DNoYAzzPPUA1qm/XXGVdSbSYvdfDhDzYp5QlZ0UKZ+hy+Uy3sAM4NTDd5VR5uN/BWH84IZrO4837sKBSCO3rCE7yxpQf0qRqjkKxqId6csLRcUspPbkhPhLP0epKolRMoU+wwrNFJ2orn3qdEQgenwQn1bJesmaSCs/S04kz7qooxurjK5uONehDDKp6ILY4pDD2ivAymuAu6t3B1r6wPIfIsz8PnKNNNB6nD2I1R10OaK7ySBI1HUOsqQ9OcZqij65VkyUOCKueUpeqQcG5DLs/O6h6Er4wbNHVfCjAVbE4SPOUj132uTx60Z6/D+i0HmqllilbsEEaaVwTCZWI8mBxarIOTpOmYe1W+fgPkyO7vzDQ1grq7Wk6EICmJw6DIsukbiYnK3Z/VnDkUUibUdylufS/Kep8mFNy7cg7Q8osD5usTBMVc4fJCQ+Ak+Qy2ScJnHuvbHCcDxCGhoTqbQ1T2sbfFJTcf6Lfu47P8HjTxZ3h1DXGHOUowNx/or/C1vJ3ACuHASu+jPXYcJW8uQt8ibDTYKRlr9+7ro2wu5YX7IuPwM4wVkvOrzXHYm8gK79WONucnh1BTSNwhqMusxfYeFp8HvqzgA9FZ34FZ/GwHZNfsxfUvTFvahST0RnqkR3SxlSfz13KPnHdOfHlD2AGkwkdjLxaOtCc7G6+fYcX+FgEM2ekObjAD2DggL95aLmBfrHhMI8GZkVQBUyMD9O4VD7wRmY0e7HNUcs278ol6+5/cUTrFMpSk3oCafe5vUslmim/nqlCJpa43PAUHTJOZBcRu/HuhgkoO44wtGBOqnlvCerQokX9RCXE/Jlfu06ZrmdkZi+qZw2xMjRdlXjq9cPOADk4172Y1n5Ny14u8Wn2Bpp+iIWOxeyF9WSc7f7cQuCenpbou7FPf7Gt9SqfFvSgkp6Mb3d/D7v4rLSS//6Vt6vyl37xUTQSXaFMVPSsyn9Aveuq/QX7KvTEHzANtp0fjsi+PDih2kvaJ9bzzqD9yq0ByLx3edgGpIT5i3Po6GNt+bRCNJl43neRPtjXe8vFE+r4sYDxJ1vBIQ+sbYxvoOvdlmWyHzdT0LzysWcLdQscF7wekA3X3X8iHcNM4+AFUq/ciUfTFN8b5bwsE+r8fZchE+0V/Q5z19e+naIhK6+/SG/GjZd8cpDm/rVp/qbNZpPxm1o5VoDtmLnjHq8p7wXqQsY3t6IylJwhJceCCfvilk8DvP7yf1BLAwQUAAAACABXIXVdn7wLN2YAAADXAAAAEwAAAHB1Ymxpc2hlZF9maWxscy5jc3ZNzTsOg0AMhOGes7BoPCu/eloOQEnBDXJ/BbE4xOWn0e/1+JzztkMxEUCDN8jcZEmGdOqj8aj1pBfmD82zpoIKIBJjyzegJmSp8Na+IDM4CvoWwmkxtv73jVLfLhoFu5QpVNwn0xdQSwECFAMUAAAACABXIXVdySvKcX0OAABnKQAADwAAAAAAAAAAAAAAgAEAAAAAb3RoZXJfd2VsbHMuY3N2UEsBAhQDFAAAAAgAVyF1XZ+8CzdmAAAA1wAAABMAAAAAAAAAAAAAAIABqg4AAHB1Ymxpc2hlZF9maWxscy5jc3ZQSwUGAAAAAAIAAgB+AAAAQQ8AAAAA',
    'grace': 'UEsDBBQAAAAIAFchdV2huC1q+wUAAA0PAAALAAAAc3RvcmFnZS5jc3ZtVz3ObTcO67OWPEP/lvpps4CUKWYHs38Mbcs+3wPSXsLXMkWROv/553///fOvv8npDyGSX2S/iP/8xVGDIlNFa1pQWcN+YJchk6ZGlKn6g2PBxiMsWNVE0tmjwblAkRpKM3hyTm8k9zHRIWKWU8niA2sfmzk0KiSLQhpi2hDlEPLMSKMU0gZ5l2rDQjOTI1FQBTcq5yEM3I3cZ6JiyQ0rsAPLHO5SoeI43+8E3KczBjGJKG4IPCgb1g2LLNimVhWJml74cIxfxrR01FRs91X6OA7AZMrO6Q5C7unNMYuPAou4F38Qs2ajm2Sjif7g1oka/PvnzTPeMsAU5fy41Etz1TBWS4kI6tbpJVpxp+BBKZMt33sO00wDNa5ixSCfEmn0cFVoLqkLz6OJjdojmmowT9OCrIjLteFLtA50Ft1i9WlyDzfPDm3gbFasqo5w7NFMPMTCcSnra5I9mimGJ1nh32UJrxo+UpZ/kYc1yw7hqZbu7rxbj5bJhgeYgvCebqxJ1jlHqZFQFUbrFXxYXr0rUO1qq6+3nkOyDXQPrcsoDBBb03TVjKdGos6ZXCnNkz+SHc0NCgyRMf75MOGP5HLAhiaRLnw22iRzYDqhUycp+XHYXoemQ2sYmd/gZnl5gghNUj6K82YY1w5OBoOYslgduPceu8CAuinKOs2b9/CRMjoAEuBEMl2h5gaPliELyRSFIn+ArWUGiAZwYATJXr2tZRsEljEi7MtRuIm8Wl7NDcIM6ZugeCyHDiuGJgLleZ0piY9lNAHjpW71XRyPZptDk4partJoaxk3Jzg8WnwXH44xsgMD5OB6j+6DN88+MUGgL39z9Lg0q44i+MFv3Y2mWZTRBHi6Ekv0cMalWQt6TXLwWD6pwUMzZnqsqYYUgzC4XI22OS8HLHFcUC9G4slZkDKMN0OTOzE2GfMRDcnB8gMmGP7sdX5Ey0ACTRbb5nuUMz+iMdqrf3DoeGY2H9NByBsIXuGGM44RzifnGWOCZA6f2wqt4UO1w/blG4bZ4GE65gi8lfEqkx7veW0jC6qsiTRjEPrATTQODFvXoVla787rzegvMq7M74TNK2YfPMGEQ+p4Jv6i0RYzjULkIlrtC6l8JOuyFIBUk2/M5OMYa4LCKnDxBBunQflZRo7AMfvpkPn58hzkCPzPAvPxC50rJ2G7gDfTvDW1Y6xNAC4Iv8EcTdVGN7/KPNaVmLyM69j5bDlWgsHpKR6/eW0ZavTPoPJKGJvHChaoNyZygPuRLWFoFDOHEEDP/HvIk3CujAc7advr90vqsQsdLZHhhK5nnv+uS69g/QCJDG5PGl9Y38DzhJIIjpE9PXX5VcJIG6bACQU0EfXMwnGzJnvvYtVoMywIc4cLYXB1nb0Xb4ozR+LNAfiLkXrbhSKMMeh90Bs8XoHVA7JHQCHo/XvvYTplFHIxYF+59qkLHq5C4MkCz0bNP597mV7rEsPAnE+X1sX7f1tROeCbNYUgqZ6eBTfTOI2z2MMU+Wg7KxZ6iMbeONAhVO7GvRUttIW8lhPscYY7q61ioU00CYIRfPXUbgUs+Kxxufy8UC7q31c3Oq/vrpjHHu5mZ6Na4DFlBKcg0u0H0Qvs7KsVI0jN4Lu3ALyKhgVNSBq7pV2zX2grGqJDpRgfGMrdSxfcRCsNEMHU+bUVv4/2eyFK/KDY5M+23nATjSZj0UDdhTefFFpoK7rWklFwBVjgmodGW9Gy/ATfHvgsuWO60CZasY0ldq7PUBbYgoY7YhuDV50ckkY3zWvx8dkL7/vXo2ZTRD3q/LmjL/SwnD4wSCcgLnQ4jl0MDtytp8FDkmMlxXwqvJyeGPmTMlwKi6MkVjVu09jY++pQxXjij71X5YU2wYaa0CMkzw+xypMyvtzWLotJKihqzz3LxzC29ImYD/aXmSyflrGrQXH4DqOnx/vV96vWo3IiifwZDt/PPnY4iiE/Ch+N2o7C99MPCgdd2B7khMVs8LCMJQ9xuJ70o6RD88QyhdyCCCE37rl9335ueA2CAprxddgavR8VtboAKrzXuP8DUEsBAhQDFAAAAAgAVyF1XaG4LWr7BQAADQ8AAAsAAAAAAAAAAAAAAIABAAAAAHN0b3JhZ2UuY3N2UEsFBgAAAAABAAEAOQAAACQGAAAAAA==',
    'wells': 'UEsDBBQAAAAIAFchdV21VgmcpAYAABcUAAAKAAAAbGV2ZWxzLmNzdl1YQY4kKA6891uqUxgMNseR5jTSPmCPe9gfzP812BGG1EgtZVfaGIiwwyb//N/f///5z3/bbD9//dGa5P/nr95a+93k/Pv5rZ++fn6Pz8j/S1Nae1rXZ7Y+w7L0fPjHB+2DdnPYd7fzKe1jWhtouthH9rHMz4BH/7RNh8kTuPU4wvQeWzRx2lfa+0fOCeWzNcP03WsDC3uYfMd6s0WDl2FYGO6G+37fZgSeNEgrg+L7SRgEIJ0QGtc8h5AZf0ozHkI6PYZbrtW85rkNgJYHtLdEWsTjr7E77QW1LSBpRNKFDoNA9m0JwYpDyLlw7VBIb484fRwcZRaQUkCvAzQCzCRN5CBLj0UP3DPO2pPvUQ7GS4y8g41wvvG9MAAEuAFygh6bVDbrE59AabdyAQcHYrNkaBVYNAsjDCETnjsyG6SIOLtHQh8eXMPBZ96gf+V7nXLFX9oRoD8emuP4gPsQMcTKp9I+sjCQlOBMTphxw4CMeVDw+FAgrXWMSazWTjgTh8Oc0Fw5P6XzlDPPZDe+EadhGh8ta8e003xzv2nGUX6/C17twNczfOO5CP9hhviOyQMoN74MGBL048lA32Xv2Ll/+ly4YZ5pPOhZdrh6M1o7AfMG2D0RoVCNL8xbAtri48iId6fDRXwkJRsqc9dPmvvN7fOxP11oXzxdZsy5cOy116LVKrfHQ+Wk9uwV3gmsj0Rlprevay/gPfN6Z9pX9Jv0S4Dp0LT3Xssv6g3yieIPj3KovF+hop1yjoTVgj4KDXRGdaXilkOhv3izICfEZ9Be6Id4zSz/Iy0n3Wku7FGWx5IFfsRJpXYomV+iyV8k4P4oWo0+/GWz7jT411pe+Y7Ap6C50SqHImAhsyU1f1qFL/wn1sPcBNZLwAR7Wc/RGAqfi/9aKSttZb31Ov3V/46VCyg25scsBiIvkV5RsAfOobSX7Jz1Cdz5kpbSfdGSbYjNesGVi8dga0iaT/pruz43/yFIIiHs5ovmxQu44egzy3vdHJmPgj0y91ojBX49igNNTeqa2rt33bE4WANJnGAdzKQCvA7cR+LH0z3Zjz2zI6jVmsp884R+QPB58fWl+J41kbhPdLT1NeEISxq6sVYtL/g1Mz/oD+z7DV9dt81/Y+9WPtV5BWofPeekvtUZFqlpgnZoKwJGU6BDqY8uB/JJoXsFcEJQTa1v3EVuhNt5HW0zD6Jc//Rn54w10XXNaC701aEeAyOKDEZ/2W8Q9jRr0m6PgC2KZh2Ht71oftKTmwZ8tJToQC1DF2iofF+UI0fJSIWsXDem8l1YIrOyQI8+7Gy7bNv2kG4j63wB6FZXuSJDGU44DxKd9jta4qptV+DbWE+yo/FS4Gm+XZUNv08c8PTpilAg5+R4yJiaW68M4d8w58qcQGm6ELP3rRoup9HjdteO7opxZaF0/WX5AKRoEzF8Qt/95fhE6uX0FBMPK8W/0rwrJtxQIEWNe0F/VLe980fNaK2/4G9keVPSe3e4WT4TZHK8eInHQWY5CjW6sHH9ZWEqZ5sNdst+8zx7rO6eAO+H/dyMickYw8cmAeHAqSnyoGwlMKtnY9QQMFHOgvtB7xw2DQIT57KKMYueDp85Odqr0aNG+zESfIvNzkPOnXYjOdTs3MMyDelQ030oc4zXWRiYN/dDvu8fvBPx/YOcyhSPO81j0qEmk5HKNjuyj2Ev4B5t8Xwn98U6S4Uaj6rZMOU+Wp8ia2Oy5gtQ2oPcBV14Yp6Zua98PVpDGcJDc8CfUExpD/AYr1diksMmrZXphjdv9rl9oKe5XlHLv4f/iT4q7WEtG0/iHGsTU2lfo6TyrTlAeoP9zfAGu3QULVGX7wet6BvFV+ftLuzaOf0zgmdCizwS+kzeds83J4749Zj1DgpaZT1qVb6eswtJjbrQbw+Q4Ic2KMHYoHHeU5TmGErH8y3Uikd5RKhMJla+Z2e7m5Tgr0aQFJuxd4q87jp6Cq+vhQdIXfa+qdge2RSaMsArAsWbdOMB6jTfElCsdz7wTxx6FB+Tbas5GB9Mmf4IEfyKsjEyCq23JIB2I5I6aL8FYYkjutvCfCz91YPFaBK/D2iWgyjtV3823qmN6s/OKv0JENBbmOPPq7rsJUADg9YpLBqqGPbiwyB3mi9yaY/j1SQTNbvwo4J8vWqHoV4go05r4S/kbzk0VGuHh//oICibpOH64wt8DnAumJ2hnjK+xky2R0t4bFo5jEr2PNwsjbdZDlUNhpdD/hoWv1LUGS4FDQXF9i/jQV9CoUHPr38AUEsDBBQAAAAIAFchdV2+muxblAAAAL0AAAAMAAAAc3RhdGlvbnMuY3N2VcyxCsIwEIDhvU/hAxwhvbTajEIGLe0kIo5HG5LQtJb2MujTSxUEt3/4+G82xrOBhjhw6i00j8l9y4SVl9AxXGd6hUhg7Mw+a++ylIClkAVoKRRC+xzttIbJeThRTOQ8MWAl9D6rj1LmgIWocMN5ATWNFOe0/GJ3oZ4WQC0q/ZmXmz+ozePfvE0Dk6POewKFQqvsDVBLAQIUAxQAAAAIAFchdV21VgmcpAYAABcUAAAKAAAAAAAAAAAAAACAAQAAAABsZXZlbHMuY3N2UEsBAhQDFAAAAAgAVyF1Xb6a7FuUAAAAvQAAAAwAAAAAAAAAAAAAAIABzAYAAHN0YXRpb25zLmNzdlBLBQYAAAAAAgACAHIAAACKBwAAAAA=',
}

@lru_cache
def part(name):
    filename = name + ".zip"
    for folder in (Path("data"), Path("4. Forecasting groundwater levels/data")):
        if (folder / filename).is_file():
            return zipfile.ZipFile(folder / filename)
    return zipfile.ZipFile(io.BytesIO(base64.b64decode(portable_zips[name])))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def read_table(section, table, index="Date"):
    with part(section).open(table + ".csv") as source:
        return pd.read_csv(source, index_col=index, parse_dates=(index == "Date"))

levels = read_table("wells", "levels").asfreq("MS")
stations = read_table("wells", "stations", "WellID")
display(stations[["District", "Upazila", "Latitude", "Longitude"]])

In [ ]:
target = "MY050"
neighbor = "JA001"  # Try MY055, another well near Haluaghat.
horizon = 6
origins = pd.date_range("2010-01-01", periods=7, freq="6MS")  # first hidden month
full_backtest = False  # Enable to evaluate all seven windows.
if not full_backtest:
    origins = origins[[0, 3, 6]]  # Include both January and July starts.
example = origins[-1]

def history(origin):
    return levels.loc[levels.index < origin, [target, neighbor]].tail(120)

def seasonal_forecast(origin):
    dates = pd.date_range(origin, periods=horizon, freq="MS")
    prior_year = dates - pd.DateOffset(years=1)
    return pd.Series(levels[target].reindex(prior_year).to_numpy(), index=dates)

past = history(example)
ax = past[target].tail(36).plot(x_compat=True, marker=".", label="Observed")
seasonal_forecast(example).plot(x_compat=True, ax=ax, marker="o", label="Same month last year")
ax.axvline(example, color="gray", linestyle="--")
ax.set(ylabel="Groundwater level (m relative to ground)", title=f"{target} · predict the next six months")
ax.legend()
plt.show()

*Where was the water table deepest last year? Which month would the seasonal baseline pick this year?*

<details>
<summary>What the orange points can tell us</summary>

The orange points in the next figure show where the earlier session’s published file supplied a value. They help locate gaps. The forecast experiment separately estimates those gaps using only the 120 months before each cutoff. A random forest uses relationships among wells to estimate missing readings; the iterative imputer revisits incomplete columns as estimates become available.

At the January 2013 cutoff there are only three missing MY050 readings in ten years. A small change in the forecast would be plausible. Estimated values can restore a seasonal pattern or smooth away a genuine departure. The hidden six months decide whether either effect helped.

</details>

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Regional GRACE grid cell nearest MY050 (25.07 N, 90.36 E).
grace = read_table("grace", "storage")["MY050"].asfreq("MS")
published_fill = read_table("gap_filling", "published_fills")
all_wells = levels.join(read_table("gap_filling", "other_wells"))
# Demonstration only: the published values help visualize what the earlier session filled.
published = published_fill[target].reindex(past.index)
ax = past[target].plot(x_compat=True, label="Observed")
added = published[past[target].isna()].dropna()
ax.scatter(added.index, added, color="orange", label="Published gap fills")
ax.set_ylabel("Groundwater level (m relative to ground)")
ax.legend()
plt.show()

all_wells = all_wells[['MY050', 'DH071', 'JA001', 'MY055', 'RJ007', 'BO034', 'SY105', 'DH088', 'RJ105', 'KT010', 'CT038', 'BA016', 'DI074']]

def fill_history(origin):
    frame = all_wells.loc[all_wells.index < origin].tail(120).dropna(axis=1, how="all")
    imputer = IterativeImputer(estimator=RandomForestRegressor(
        n_estimators=30, max_features="sqrt", min_samples_leaf=5, n_jobs=1,
        random_state=42), max_iter=3, random_state=42)
    values = pd.DataFrame(imputer.fit_transform(frame), index=frame.index, columns=frame.columns)
    first, last = frame[target].first_valid_index(), frame[target].last_valid_index()
    return values[target].where((frame.index >= first) & (frame.index <= last))

filled_histories = {origin: fill_history(origin) for origin in origins}

## 13:40 · Forecast from one well
Chronos-2 learned patterns during pretraining. Here its weights stay fixed. The input is up to ten years of monthly history, and the output is six monthly forecasts. The median is our point forecast, and the 10th and 90th percentiles describe the model's central 80% prediction interval. In zero-shot mode, Darts uses `fit()` to initialize the model for the selected inputs.

*Same weights, different well history. What can change in the forecast?*

<details>
<summary>Pretraining, context and the shaded band</summary>

A time series foundation model learns forecasting patterns across many series during pretraining. Loading the checkpoint gives us those learned weights. Supplying ten years of MY050 readings gives the model local context. A different history can therefore produce a different forecast with exactly the same weights.

Each forecast month has its own median and interval. The shaded band runs from the model’s 10th to its 90th percentile for that month. Its nominal coverage is 80% per month. Six observations provide little evidence about calibration, and the band makes no promise that the entire six-month path lies inside it.

</details>

In [ ]:
import torch
from darts import TimeSeries
from darts.models import Chronos2Model
from darts.utils.likelihood_models import QuantileRegression
from darts.utils.timeseries_generation import datetime_attribute_timeseries

if model_name == "Chronos-2 Small":
    model_class, model_id = Chronos2Model, "autogluon/chronos-2-small"
elif model_name == "TiRex-2":
    from darts.models import TiRex2Model
    model_class, model_id = TiRex2Model, "NX-AI/TiRex-2"
else:
    raise ValueError("Choose Chronos-2 Small or TiRex-2 in the setup cell.")

device = "cuda" if torch.cuda.is_available() else "cpu"
quantiles = [0.1, 0.5, 0.9]

def make_model(**options):
    return model_class(input_chunk_length=(1, 120), output_chunk_length=horizon,
                       hub_model_name=model_id, likelihood=QuantileRegression(quantiles),
                       pl_trainer_kwargs={"accelerator": "gpu" if device == "cuda" else "cpu",
                                          "devices": 1}, **options)

def inputs(origin, mode):
    past = history(origin)
    series = TimeSeries.from_series(past[target].astype("float32"))
    if mode in ("filled", "grace"):
        series = TimeSeries.from_series(filled_histories[origin].astype("float32"))
        if mode == "filled":
            return series, {}
        # Assumed two-month publication lag; historical release dates are unavailable.
        observed = grace.shift(2).reindex(past.index).astype("float32")
        assert observed.notna().any(), "No GRACE observations in this history"
        return series, {"past_covariates": TimeSeries.from_series(observed)}
    if mode == "neighbor":
        return series, {"past_covariates": TimeSeries.from_series(past[neighbor].astype("float32"))}
    if mode == "calendar":
        calendar = datetime_attribute_timeseries(series, "month", cyclic=True,
                                                 add_length=horizon, dtype="float32")
        return series, {"future_covariates": calendar}
    if mode == "target":
        return series, {}
    raise ValueError(f"Unknown input mode: {mode}")

models = {}
forecast_times = []
def forecast(origin, mode="target", tuned=None):
    started = perf_counter()
    series, covariates = inputs(origin, mode)
    if tuned is None:
        if mode not in models:
            # Initialize each input schema once using only pre-evaluation history.
            initial, initial_cov = inputs(origins[0], mode)
            model = make_model()
            model.fit(initial, **initial_cov)
            models[mode] = model
        model = models[mode]
    else:
        model = tuned
    pred = model.predict(horizon, series=series, **covariates, predict_likelihood_parameters=True)
    dates = pd.date_range(origin, periods=horizon, freq="MS")
    assert pred.time_index.equals(dates)
    forecast_times.append(perf_counter() - started)
    return pd.DataFrame(pred.values(), index=dates, columns=[str(q) for q in quantiles])

single = forecast(example)
display(single.round(2))

## 13:55 · Compare gap filling and satellite information
Compare original well readings, cutoff-specific gap-filled readings, and the same gap-filled readings with GRACE. The satellite input is a regional terrestrial water-storage anomaly at the nearest grid cell. We assume two months of reporting delay for this retrospective exercise. The imputer uses only the 120 months preceding each forecast.

*The orange points are estimates. Which comparison isolates their contribution? Which isolates the contribution of regional water storage?*

<details>
<summary>Why the three runs answer two different questions</summary>

| Compare | What changes | What the result tests |
|---|---|---|
| Original history → gap-filled history | Estimated readings replace internal gaps | Whether the imputation procedure helps the forecast |
| Gap-filled history → gap-filled history + GRACE | A second historical series becomes available | Whether regional storage adds information after gap filling |

GRACE measures a regional anomaly in total terrestrial water storage, which includes groundwater and other water stores. MY050 measures water level at a point. A shared seasonal pattern could help, while regional averaging could hide a local change. The model comparison measures predictive value; it cannot establish that GRACE explains the cause of a change at the well.

With the assumed two-month delay, the December 2012 input row carries the October 2012 GRACE value. The calendar exercise makes a useful contrast: the months January–June 2013 were already known at issue time, so their calendar features can extend through the horizon.

</details>

In [ ]:
filled_pred = forecast(example, "filled")
grace_pred = forecast(example, "grace")
ax = past[target].tail(24).plot(x_compat=True, color="black", label="Target history")
for pred, label in [(single, "Target only"), (filled_pred, "Gap-filled"), (grace_pred, "Gap-filled + GRACE")]:
    line, = ax.plot(pred.index, pred["0.5"], marker="o", label=label)
    ax.fill_between(pred.index, pred["0.1"], pred["0.9"], color=line.get_color(), alpha=0.12)
ax.set(ylabel="Groundwater level (m relative to ground)", title=f"Forecast at {target} using history through {past.index[-1]:%B %Y}")
ax.legend()
plt.show()

## 14:15 · Reveal the observations and test accuracy
Compare mean absolute error (MAE) in metres. An error of 0.5 m counts equally whether the forecast was too shallow or too deep. Score all methods on the same observed months with an available seasonal baseline. Then repeat across the selected non-overlapping six-month windows.

*Does the lowest average error also mean the lowest error in each six-month window?*

<details>
<summary>Reading the error table in metres</summary>

If the observed level is −6.0 m and the forecast is −5.4 m, the model predicted a water table 0.6 m too shallow. Its absolute error is 0.6 m. Averaging these distances gives MAE in the same units as the well reading.

Subtract the **Gap-filled** MAE from **Target only** to measure the benefit of filling gaps. Then subtract **Gap-filled + GRACE** from **Gap-filled** to measure the incremental benefit of GRACE. A positive difference means a reduction in error. For example, a drop from 0.80 to 0.65 m would save 15 cm of absolute error per scored month on average. Those numbers illustrate the calculation; the output below supplies the actual values.

Each method is scored on the same months. A missing observation or missing seasonal reference can reduce the count. The pooled MAE weights every scored month equally, so a window with more observations contributes more.

</details>

In [ ]:
def comparison(origin, single, filled_pred, grace_pred):
    table = pd.DataFrame({"Seasonal": seasonal_forecast(origin), "Target only": single["0.5"],
                          "Gap-filled": filled_pred["0.5"], "Gap-filled + GRACE": grace_pred["0.5"]})
    table["Observed"] = levels[target].reindex(table.index)
    return table

shown = comparison(example, single, filled_pred, grace_pred)
ax = shown.plot(marker="o", ylabel="Groundwater level (m relative to ground)",
                title=f"{target} · forecasts and observations")
plt.show()
display(shown.round(2))

In [ ]:
windows = {}
for origin in origins:
    if origin == example:
        windows[origin] = shown
    else:
        windows[origin] = comparison(origin, forecast(origin),
                                     forecast(origin, "filled"), forecast(origin, "grace"))

results = pd.concat(windows, names=["Forecast start", "Month"]).dropna()
errors = results.drop(columns="Observed").sub(results["Observed"], axis=0).abs()
by_window = errors.groupby(level="Forecast start").mean()
by_window["Months scored"] = errors.groupby(level="Forecast start").size()
display(by_window.round(2))
display(errors.mean().rename("MAE across scored months (m)").round(2))

print(f"{model_id} on {device} | package setup: {install_seconds:.1f} s")
print(f"{len(forecast_times)} forecasts including first model loads: {sum(forecast_times):.1f} s")

## 14:35 · Discuss the result
Did filling gaps lower MAE? Did adding GRACE improve the same filled forecast? Check each window before interpreting the average.

Optional: call `forecast(example, "neighbor")` or `forecast(example, "calendar")` to try neighboring-well history or a known-future calendar on the original target history.

**Across benchmarks** · [TSFM.ai benchmark archive](https://tsfm.ai/benchmarks). These examples were checked on November 21, 2026.

| Benchmark | Task | Leading entry on the displayed metric |
|---|---|---|
| FEV Bench | Point forecast across real-world tasks | TimeRSI-1.5, 41.44 MASE skill score (higher is better) |
| GIFT-Eval | Point forecast rank across diverse settings | Albert, 16.45 average MASE rank (lower is better) |
| QuitoBench | Changing trend and seasonality in application traffic | CrossFormer, 2.86 mean rank, followed by full Chronos-2 at 3.36 |
| BOOM | Probabilistic forecasts of infrastructure telemetry | Toto 2.0 variants occupy the first three places |
| Impermanent | Forecasts scored as new observations arrive | ZeroModel, 2.21 mean MASE/CRPS rank |
| ARFBench | Answering questions about anomalies in charts | Toto-1.0-QA-Experimental 32B, 63.9% accuracy among individual models |
| TIME | Performance by task and temporal pattern | Select a task and metric on the official leaderboard |

*FEV Bench also uses a seasonal baseline. How much would a result on application traffic tell us about the next six months at a Bangladesh well?*

<details>
<summary>Interpreting different outcomes</summary>

If the seasonal baseline wins, check whether it captured the timing of the deepest month or just its approximate level. If GRACE helps in one window and hurts in another, inspect the individual months before generalizing. If the medians look almost identical, compare their absolute errors at the turning point.

Three forecast windows at one well establish what happened in this exercise. A stronger transfer test would keep the horizon and scoring rules fixed and repeat at another well, using the GRACE cell matched to that location. The compact satellite file in this notebook is specific to MY050.

</details>

### Optional · Fine-tune on earlier groundwater records
On a GPU, enable the switch below. Use readings strictly before the earliest evaluation window to update a copy of the model. Compare it with the frozen target-only model on exactly the same scored months.

*In the zero-shot runs the weights stayed fixed. Here, which observations can update them?*

<details>
<summary>What changes during fine-tuning</summary>

This optional run uses earlier contiguous stretches of observed groundwater history and updates the pretrained weights for one epoch. The frozen and adapted models then receive the same target-only inputs at the same evaluation dates. Comparing those two errors isolates the effect of local training. A decrease is evidence of improvement on these dates; an increase is also informative when the local training record is limited.

</details>

In [ ]:
run_fine_tuning = False
if run_fine_tuning:
    if device != "cuda":
        raise RuntimeError("Choose a GPU runtime for the optional fine-tuning exercise.")
    training = read_table("fine_tuning", "earlier_wells")
    training = training.loc[training.index < origins[0]]
    if model_name != "Chronos-2 Small":
        raise ValueError("Select Chronos-2 Small for this fine-tuning exercise.")
    # Use only observed, contiguous training stretches with sufficient history.
    stretches = []
    for _, well in training.items():
        for _, segment in well.groupby(well.isna().cumsum()):
            segment = segment.dropna()
            if len(segment) >= 120 + horizon:
                stretches.append(TimeSeries.from_series(segment.astype("float32")))
    tuned = make_model(enable_finetuning=True, n_epochs=1, batch_size=4)
    tuned.fit(stretches)
    tuned_values = pd.concat({o: forecast(o, tuned=tuned)["0.5"] for o in origins},
                             names=["Forecast start", "Month"]).reindex(results.index)
    assert tuned_values.notna().all()
    display(pd.Series({"Frozen": errors["Target only"].mean(),
                       "Fine-tuned": (tuned_values - results["Observed"]).abs().mean()},
                      name="Target-only MAE (m)").round(2))

**Resources** · [Chronos forecasting examples](https://github.com/amazon-science/chronos-forecasting) · [Chronos-2 model card](https://huggingface.co/amazon/chronos-2) · [Darts foundation-model examples](https://unit8co.github.io/darts/examples/25-FoundationModel-examples.html)